# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/susheel123-sketch/Flyrank-Internship-ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of analysis:** One row = one content page, aggregated over one calendar month
(daily rows in `fact_content_daily_performance` are summed/averaged per `content_hash_id`
to form a page-month snapshot).

**Time window:** `month=2026-03` for the current snapshot, `month=2026-02` for computing
month-over-month change (needed to build the trend proxy, since no `trend_direction`
column exists in this raw data). The final month (June 2026 / `_sample`) is a sealed
outcome window and is not touched here.

In [2]:
agg_march = perf_march.groupby("content_hash_id").agg(
    gsc_impressions=("gsc_impressions", "sum"),
    gsc_clicks=("gsc_clicks", "sum"),
    gsc_avg_position=("gsc_avg_position", "mean"),
    ga4_engaged_sessions=("ga4_engaged_sessions", "sum"),
    scroll_events=("scroll_events", "sum"),
).reset_index()

print(f"Daily rows in March: {len(perf_march)}")
print(f"Unique content pages after monthly aggregation: {agg_march['content_hash_id'].nunique()}")
agg_march.head(3)

Daily rows in March: 9841378
Unique content pages after monthly aggregation: 331437


,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_engaged_sessions,scroll_events
0,content_000005d4ced12088,86,0,72.854861,0.0,0.0
1,content_00001e488b74b799,0,0,NaN,0.0,0.0
2,content_00007bd2985b77c3,47,0,5.269565,0.0,0.0


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Feature:** `gsc_impressions`, `gsc_clicks`, `gsc_avg_position`, `ga4_engaged_sessions`,
`scroll_events` (all aggregated over March), plus `word_count`, `content_type`,
`search_volume` from `dim_content`.

**Label/proxy:** `is_declining` — built by comparing March `gsc_impressions` to February
`gsc_impressions` for the same page; `True` if impressions dropped more than 10%.

**Context:** `content_hash_id`, `client_hash_id`, `report_date`/month — identify the row,
not used as signals.

**Excluded:** `last_optimized_date`, `optimization_eligible_date` — excluded because
these reflect whether a review action was already taken, which is downstream of the
decision this model supports, not a predictor available before it.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [3]:
# Grain check
dupes = agg_march["content_hash_id"].duplicated().sum()
print(f"Duplicate content_hash_id after aggregation: {dupes}")
assert dupes == 0
# Row count + date span
print(f"Pages in March aggregate: {len(agg_march)}")
print(f"Distinct report_dates in March: {perf_march['report_date'].nunique()}")
print(f"Date range: {perf_march['report_date'].min()} to {perf_march['report_date'].max()}")
# Availability check (IS TRUE equivalent)
available = perf_march[perf_march["gsc_data_available"] == True]
print(f"Daily rows before availability filter: {len(perf_march)}")
print(f"Daily rows after gsc_data_available filter: {len(available)}")

Duplicate content_hash_id after aggregation: 0
Pages in March aggregate: 331437
Distinct report_dates in March: 31
Date range: 2026-03-01 to 2026-03-31
Daily rows before availability filter: 9841378
Daily rows after gsc_data_available filter: 3611061


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**What this data can never tell you:**
- `ga4_data_available` is `None`/False for many clients (only `client_has_gsc=True`
  in the sample rows shown) — engagement-based features are missing or unreliable
  for GSC-only clients.
- The trend proxy compares only two months (Feb→March); it can't distinguish a real
  sustained decline from ordinary month-to-month noise.
- `client_created_date` varies widely — newer clients have short history, so their
  month-over-month comparison is based on less established data than long-tracked clients.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.